# IndicConformer 600M — Odia ASR Fine-tuning

**Model**: [ai4bharat/indic-conformer-600m-multilingual](https://huggingface.co/ai4bharat/indic-conformer-600m-multilingual)  
**Data**: [OpenSLR-103](https://www.openslr.org/103/) — MUCS 2021 Odia (~94h train, ~5.5h test)  
**Storage**: Google Drive (all checkpoints + data saved there)

**Runtime**: Set to `T4 GPU` (Runtime → Change runtime type → T4 GPU)

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_DIR = '/content/drive/MyDrive/odia_asr'
os.makedirs(DRIVE_DIR, exist_ok=True)
os.makedirs(f'{DRIVE_DIR}/data/raw', exist_ok=True)
os.makedirs(f'{DRIVE_DIR}/data/manifests', exist_ok=True)
os.makedirs(f'{DRIVE_DIR}/outputs', exist_ok=True)
print('Drive mounted. Working dir:', DRIVE_DIR)

## 2. Install Dependencies

In [ ]:
!pip install -q transformers>=4.49.0 huggingface_hub torchaudio
# IndicConformer uses onnxruntime for inference but we train with PyTorch directly
!pip install -q onnxruntime
print('Dependencies installed.')

## 3. Download Odia Dataset from OpenSLR-103

In [ ]:
import os, tarfile, urllib.request

RAW_DIR = f'{DRIVE_DIR}/data/raw'

URLS = {
    'Odia_train.tar.gz': 'https://openslr.elda.org/resources/103/Odia_train.tar.gz',
    'Odia_test.tar.gz':  'https://openslr.elda.org/resources/103/Odia_test.tar.gz',
}

def download(url, dest):
    if os.path.exists(dest):
        print(f'  Already downloaded: {dest}')
        return
    print(f'  Downloading {os.path.basename(dest)} ...')
    urllib.request.urlretrieve(url, dest)
    print(f'  Done: {dest}')

def extract(tar_path, out_dir):
    name = os.path.basename(tar_path).replace('.tar.gz', '')
    if os.path.isdir(os.path.join(out_dir, name)):
        print(f'  Already extracted: {name}')
        return
    print(f'  Extracting {tar_path} ...')
    with tarfile.open(tar_path, 'r:gz') as t:
        t.extractall(out_dir)
    print(f'  Extracted to {out_dir}/{name}')

for fname, url in URLS.items():
    dest = os.path.join(RAW_DIR, fname)
    download(url, dest)
    extract(dest, RAW_DIR)

print('\nData ready in:', RAW_DIR)
!ls -lh {RAW_DIR}

## 4. Inspect Dataset Structure

In [ ]:
import glob

for split in ['Odia_train', 'Odia_test']:
    split_dir = os.path.join(RAW_DIR, split)
    print(f'\n=== {split} ===')
    for root, dirs, files in os.walk(split_dir):
        rel = os.path.relpath(root, split_dir)
        indent = '  ' * rel.count(os.sep)
        print(f'{indent}{rel}/')
        for f in files[:5]:
            print(f'{indent}  {f}')
        if len(files) > 5:
            print(f'{indent}  ... ({len(files)} files total)')
        break  # only top level

## 5. Build JSON Manifests

In [ ]:
import json, wave, glob

MANIFEST_DIR = f'{DRIVE_DIR}/data/manifests'
os.makedirs(MANIFEST_DIR, exist_ok=True)

def get_duration(wav_path):
    try:
        with wave.open(wav_path, 'r') as wf:
            return wf.getnframes() / wf.getframerate()
    except:
        return 0.0

def find_transcription_file(split_dir):
    """Try common transcription file names; fall back to any .txt in the dir."""
    for name in ['transcription.txt', 'transcriptions.txt',
                 'transcript.txt', 'text', 'transcription.tsv']:
        path = os.path.join(split_dir, name)
        if os.path.exists(path):
            return path
    txts = glob.glob(os.path.join(split_dir, '*.txt'))
    if txts:
        return txts[0]
    raise FileNotFoundError(f'No transcription file found in {split_dir}')

def load_transcriptions(split_dir):
    trans_file = find_transcription_file(split_dir)
    trans = {}
    with open(trans_file, encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line: continue
            parts = line.split('\t', 1) if '\t' in line else line.split(' ', 1)
            if len(parts) == 2:
                trans[parts[0].strip()] = parts[1].strip()
    print(f'  Transcriptions: {trans_file} ({len(trans)} entries)')
    return trans

def build_manifest(split_dir, out_path):
    trans     = load_transcriptions(split_dir)
    audio_dir = os.path.join(split_dir, 'audio')
    if not os.path.isdir(audio_dir):
        audio_dir = split_dir
    wavs = sorted(glob.glob(os.path.join(audio_dir, '*.wav')))
    print(f'  WAV files: {len(wavs)}')

    records, skipped = [], 0
    for wav_path in wavs:
        fid  = os.path.splitext(os.path.basename(wav_path))[0]
        text = trans.get(fid)
        if text is None:
            skipped += 1
            continue
        records.append({
            'audio_filepath': os.path.abspath(wav_path),   # absolute path for portability
            'text':           text,
            'duration':       round(get_duration(wav_path), 3),
        })

    with open(out_path, 'w', encoding='utf-8') as f:
        for r in records:
            f.write(json.dumps(r, ensure_ascii=False) + '\n')

    total_h = sum(r['duration'] for r in records) / 3600
    print(f'  Written {len(records)} records ({total_h:.2f}h) → {out_path}')
    if skipped:
        print(f'  Skipped {skipped} (no transcript)')

for split in ['Odia_train', 'Odia_test']:
    split_key = split.split('_')[1].lower()   # train / test
    split_dir = os.path.join(RAW_DIR, split)
    out_path  = os.path.join(MANIFEST_DIR, f'odia_{split_key}_manifest.json')
    print(f'\n[{split}]')
    if os.path.exists(out_path):
        print(f'  Already exists: {out_path}')
    else:
        build_manifest(split_dir, out_path)

print('\nManifests ready:')
!ls -lh {MANIFEST_DIR}


## 6. Define Odia Tokenizer

In [ ]:
ODIA_VOCAB_RAW = (
    ['<blank>', '<unk>', ' '] +
    list('ଅଆଇଈଉଊଋଌଏଐଓଔ') +
    list('କଖଗଘଙଚଛଜଝଞଟଠଡଢଣତଥଦଧନପଫବଭମଯରଲଳୱଶଷସହ') +
    list('ାିୀୁୂୃୄେୈୋୌ\u200c\u200d୍ଂଃଁ') +
    list('୦୧୨୩୪୫୬୭୮୯') +
    list('0123456789') +
    list(',.?!।')
)
seen = set()
ODIA_VOCAB = []
for c in ODIA_VOCAB_RAW:
    if c not in seen:
        ODIA_VOCAB.append(c)
        seen.add(c)

class OdiaTokenizer:
    def __init__(self):
        self.vocab = ODIA_VOCAB
        self.c2i = {c: i for i, c in enumerate(self.vocab)}
        self.i2c = {i: c for i, c in enumerate(self.vocab)}
        self.blank_id = 0
        self.unk_id   = 1
    def encode(self, text):
        return [self.c2i.get(c, self.unk_id) for c in text]
    def decode(self, ids):
        chars, prev = [], None
        for idx in ids:
            if idx == self.blank_id: prev = None; continue
            if idx != prev: chars.append(self.i2c.get(idx, ''))
            prev = idx
        return ''.join(chars)
    def __len__(self):
        return len(self.vocab)

tok = OdiaTokenizer()
print(f'Vocab size: {len(tok)}')
test_text = 'ଓଡ଼ିଆ ଭାଷା'
print(f'Encode test: {tok.encode(test_text)}')
print(f'Decode test: {tok.decode(tok.encode(test_text))}')

## 7. Dataset & DataLoader

In [ ]:
import torch, torchaudio
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

class OdiaASRDataset(Dataset):
    def __init__(self, manifest_path, tokenizer,
                 target_sr=16000, source_sr=8000,
                 max_secs=30.0, min_secs=0.5):
        self.tokenizer = tokenizer
        self.resampler = torchaudio.transforms.Resample(source_sr, target_sr)
        self.target_sr = target_sr
        self.source_sr = source_sr

        self.data = []
        with open(manifest_path, encoding='utf-8') as f:
            for line in f:
                r = json.loads(line.strip())
                if min_secs <= r.get('duration', 999) <= max_secs:
                    self.data.append(r)
        print(f'  Loaded {len(self.data)} samples from {manifest_path}')

    def __len__(self): return len(self.data)

    def __getitem__(self, idx):
        r = self.data[idx]
        wav, sr = torchaudio.load(r['audio_filepath'])
        wav = wav.mean(0, keepdim=True)
        if sr == self.source_sr:
            wav = self.resampler(wav)
        elif sr != self.target_sr:
            wav = torchaudio.transforms.Resample(sr, self.target_sr)(wav)
        wav = wav.squeeze(0)
        label = torch.tensor(self.tokenizer.encode(r['text']), dtype=torch.long)
        return wav, label

def collate_fn(batch):
    wavs, labels = zip(*batch)
    wav_lens   = torch.tensor([w.shape[0] for w in wavs], dtype=torch.long)
    label_lens = torch.tensor([l.shape[0] for l in labels], dtype=torch.long)
    padded_wavs   = nn.utils.rnn.pad_sequence(wavs,   batch_first=True)
    padded_labels = nn.utils.rnn.pad_sequence(labels, batch_first=True, padding_value=0)
    return padded_wavs, wav_lens, padded_labels, label_lens

tokenizer = OdiaTokenizer()

train_ds = OdiaASRDataset(f'{MANIFEST_DIR}/odia_train_manifest.json', tokenizer)
test_ds  = OdiaASRDataset(f'{MANIFEST_DIR}/odia_test_manifest.json',  tokenizer)

BATCH_SIZE = 8
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          collate_fn=collate_fn, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE*2, shuffle=False,
                          collate_fn=collate_fn, num_workers=2)
print(f'Train batches: {len(train_loader)} | Test batches: {len(test_loader)}')

## 8. Load IndicConformer + CTC Head

In [ ]:
import os
from transformers import AutoModel

HF_TOKEN = ''  # Set your HF token if needed (model requires accepting terms)
MODEL_ID  = 'ai4bharat/indic-conformer-600m-multilingual'

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

print('Loading base model (this may take ~2 min)...')
kwargs = {'trust_remote_code': True}
if HF_TOKEN:
    kwargs['token'] = HF_TOKEN
base_model = AutoModel.from_pretrained(MODEL_ID, **kwargs)
print('Base model loaded.')

ENCODER_DIM = 512  # IndicConformer-600M encoder dim
VOCAB_SIZE   = len(tokenizer)

class CTCModel(nn.Module):
    def __init__(self, base, vocab_size, enc_dim):
        super().__init__()
        self.base     = base
        self.ctc_head = nn.Linear(enc_dim, vocab_size)
        self.ctc_loss = nn.CTCLoss(blank=0, reduction='mean', zero_infinity=True)

    def encode(self, wavs, wav_lens):
        # IndicConformer wraps a NeMo-style encoder
        wavs_3d = wavs.unsqueeze(1)  # (B,1,T)
        try:
            enc, enc_len = self.base.encoder(audio_signal=wavs_3d, length=wav_lens)
        except TypeError:
            enc, enc_len = self.base.encoder(wavs_3d, wav_lens)
        # enc: (B, D, T') or (B, T', D)
        if enc.shape[1] != enc_len.max():
            enc = enc.transpose(1, 2)  # → (B, T', D)
        return enc, enc_len

    def forward(self, wavs, wav_lens):
        enc, enc_len = self.encode(wavs, wav_lens)
        logits    = self.ctc_head(enc)            # (B, T', V)
        log_probs = logits.log_softmax(-1).transpose(0, 1)  # (T', B, V)
        return log_probs, enc_len

    def loss(self, lp, il, tgt, tl):
        return self.ctc_loss(lp, tgt, il, tl)

model = CTCModel(base_model, VOCAB_SIZE, ENCODER_DIM).to(device)

# Freeze encoder initially
for p in model.base.encoder.parameters():
    p.requires_grad = False
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f'Trainable params: {trainable:,} / {total:,} (encoder frozen)')

## 9. Training Configuration

In [ ]:
import numpy as np

CFG = dict(
    max_epochs           = 30,
    freeze_encoder_epochs= 5,
    learning_rate        = 1e-4,
    lr_encoder           = 1e-5,   # lower LR when encoder unfrozen
    grad_accum           = 4,
    warmup_steps         = 500,
    log_interval         = 50,
    eval_every_n_epochs  = 1,
    output_dir           = f'{DRIVE_DIR}/outputs',
)

print('Config:')
for k, v in CFG.items(): print(f'  {k}: {v}')

## 10. Evaluation (WER)

In [ ]:
def wer(preds, refs):
    total_w, total_e = 0, 0
    for p, r in zip(preds, refs):
        pw, rw = p.split(), r.split()
        total_w += len(rw)
        d = list(range(len(pw)+1))
        for ri, rch in enumerate(rw):
            nd = [ri+1] + [0]*len(pw)
            for pi, pch in enumerate(pw):
                nd[pi+1] = d[pi] if rch==pch else 1+min(d[pi], d[pi+1], nd[pi])
            d = nd
        total_e += d[-1]
    return total_e / max(total_w, 1)

def evaluate(model, loader):
    model.eval()
    preds, refs = [], []
    with torch.no_grad():
        for wavs, wl, labels, ll in loader:
            wavs, wl = wavs.to(device), wl.to(device)
            lp, el = model(wavs, wl)
            best = lp.argmax(-1).transpose(0,1)  # (B, T')
            for i in range(best.shape[0]):
                preds.append(tokenizer.decode(best[i, :el[i]].tolist()))
            for i in range(labels.shape[0]):
                refs.append(tokenizer.decode(labels[i, :ll[i]].tolist()))
    return wer(preds, refs)

print('WER function ready.')

## 11. Train!

In [ ]:
import math

os.makedirs(CFG['output_dir'], exist_ok=True)

optimizer = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=CFG['learning_rate'], weight_decay=1e-4
)

total_steps = (len(train_loader) // CFG['grad_accum']) * CFG['max_epochs']

def lr_lambda(step):
    wu = CFG['warmup_steps']
    if step < wu:
        return step / max(1, wu)
    prog = (step - wu) / max(1, total_steps - wu)
    return max(0.0, 0.5 * (1.0 + math.cos(math.pi * prog)))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
scaler    = torch.cuda.amp.GradScaler(enabled=(device.type=='cuda'))

best_wer_val = float('inf')
global_step  = 0

for epoch in range(1, CFG['max_epochs'] + 1):

    # Unfreeze encoder after freeze period
    if epoch == CFG['freeze_encoder_epochs'] + 1:
        for p in model.base.encoder.parameters():
            p.requires_grad = True
        optimizer = torch.optim.AdamW(model.parameters(),
                                      lr=CFG['lr_encoder'], weight_decay=1e-4)
        scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
        scaler    = torch.cuda.amp.GradScaler(enabled=(device.type=='cuda'))
        print(f'\n>>> Epoch {epoch}: Encoder unfrozen (lr={CFG["lr_encoder"]:.1e}) <<<')

    model.train()
    epoch_loss = 0.0
    optimizer.zero_grad()

    for step, (wavs, wl, labels, ll) in enumerate(train_loader, 1):
        wavs   = wavs.to(device)
        wl     = wl.to(device)
        labels = labels.to(device)
        ll     = ll.to(device)

        with torch.cuda.amp.autocast(enabled=(device.type=='cuda')):
            lp, el = model(wavs, wl)
            loss   = model.loss(lp, el, labels, ll) / CFG['grad_accum']

        scaler.scale(loss).backward()
        epoch_loss += loss.item() * CFG['grad_accum']

        if step % CFG['grad_accum'] == 0:
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            optimizer.zero_grad()
            global_step += 1

        if step % CFG['log_interval'] == 0:
            print(f'  Ep{epoch} [{step}/{len(train_loader)}] '
                  f'loss={epoch_loss/step:.4f} '
                  f'lr={scheduler.get_last_lr()[0]:.2e}')

    avg_loss = epoch_loss / len(train_loader)
    print(f'\nEpoch {epoch} — avg loss: {avg_loss:.4f}')

    if epoch % CFG['eval_every_n_epochs'] == 0:
        wer_val = evaluate(model, test_loader)
        print(f'  Test WER: {wer_val*100:.2f}%')

        ckpt = os.path.join(CFG['output_dir'], f'ep{epoch}_wer{wer_val*100:.1f}.pt')
        torch.save({
            'epoch': epoch, 'wer': wer_val,
            'model_state': model.state_dict(),
            'config': CFG,
        }, ckpt)
        print(f'  Saved: {ckpt}')

        if wer_val < best_wer_val:
            best_wer_val = wer_val
            best = os.path.join(CFG['output_dir'], 'best_model.pt')
            torch.save(model.state_dict(), best)
            print(f'  *** Best WER: {best_wer_val*100:.2f}% → {best}')

print(f'\nTraining complete. Best WER: {best_wer_val*100:.2f}%')

## 12. Quick Inference Test

In [ ]:
# Load best model and transcribe a test sample
best_ckpt = os.path.join(CFG['output_dir'], 'best_model.pt')
model.load_state_dict(torch.load(best_ckpt, map_location=device))
model.eval()

# Pick first test sample
wav, label = test_ds[0]
ref = tokenizer.decode(label.tolist())

wav_t  = wav.unsqueeze(0).to(device)
wl_t   = torch.tensor([wav.shape[0]], dtype=torch.long).to(device)

with torch.no_grad():
    lp, el = model(wav_t, wl_t)
    ids = lp.argmax(-1).squeeze(1)[:el[0]].tolist()
    pred = tokenizer.decode(ids)

print('Reference:', ref)
print('Predicted:', pred)

## 13. Push Fine-tuned Model to HuggingFace (Optional)

Fill in `HF_TOKEN` and `HF_REPO` below to push the trained weights.

In [ ]:
from huggingface_hub import HfApi

HF_TOKEN = ''   # your HF write token
HF_REPO  = 'shantipriya/indic-conformer-odia-finetuned'

if HF_TOKEN:
    api = HfApi(token=HF_TOKEN)
    try:
        api.create_repo(HF_REPO, exist_ok=True)
    except Exception:
        pass
    api.upload_file(
        path_or_fileobj=best_ckpt,
        path_in_repo='best_model.pt',
        repo_id=HF_REPO,
        commit_message='Upload fine-tuned Odia ASR model'
    )
    print(f'Uploaded to https://huggingface.co/{HF_REPO}')
else:
    print('Set HF_TOKEN to push to HuggingFace.')

## 14. Evaluate on IWSLT 2026 Odia (or-hi-eng)

The repo `shashwatup9k/iwslt2026_or-hi-eng` follows the same layout as other  
IWSLT 2026 tracks: `stamped.tsv` + `wav/` + `txt/` per split.

**Steps:**
1. Clone the dataset repo (or copy from Drive if already downloaded)
2. Upload `evaluate_iwslt2026.py` from your local machine
3. Run ASR evaluation (WER / CER) and inspect predictions

In [ ]:
import os, subprocess

# OdiaGenAI/iwslt-odia-speech  — layout: train/ and test_set/
IWSLT_DIR = '/content/drive/MyDrive/odia_asr/iwslt-odia-speech'

if not os.path.isdir(IWSLT_DIR):
    print('Cloning OdiaGenAI/iwslt-odia-speech dataset repo...')
    subprocess.run(
        ['git', 'clone', '--depth', '1',
         'https://github.com/OdiaGenAI/iwslt-odia-speech', IWSLT_DIR],
        check=True
    )
    print('Cloned to:', IWSLT_DIR)
else:
    print('Already present:', IWSLT_DIR)

# List available splits and their contents
print('\nAvailable splits:')
for name in sorted(os.listdir(IWSLT_DIR)):
    p = os.path.join(IWSLT_DIR, name)
    if os.path.isdir(p) and name not in ['.git']:
        tsv      = os.path.join(p, 'stamped.tsv')
        audio    = os.path.join(p, 'audio')
        or_txt   = os.path.join(p, 'or', 'txt.or')
        root_txt = os.path.join(p, 'txt.or')
        n_wav    = len([f for f in (os.listdir(audio) if os.path.isdir(audio) else [])
                        if f.endswith('.wav')])
        n_lines  = 0
        for tp in [or_txt, root_txt]:
            if os.path.exists(tp):
                with open(tp) as f: n_lines = sum(1 for l in f if l.strip())
                break
        print(f'  {name}/  stamped.tsv={os.path.exists(tsv)}  '
              f'audio_wavs={n_wav}  txt_lines={n_lines}')


In [ ]:
import math, glob
from collections import Counter
from pathlib import Path
from typing import List, Dict, Optional

# ── Metrics ──────────────────────────────────────────────────

def _edit(a, b):
    d = list(range(len(b)+1))
    for ca in a:
        nd = [d[0]+1] + [0]*len(b)
        for j, cb in enumerate(b):
            nd[j+1] = d[j] if ca==cb else 1+min(d[j], d[j+1], nd[j])
        d = nd
    return d[-1]

def compute_wer(preds, refs):
    tw, te = 0, 0
    for p, r in zip(preds, refs):
        rw = r.split(); tw += len(rw); te += _edit(p.split(), rw)
    return te / max(tw, 1)

def compute_cer(preds, refs):
    tc, te = 0, 0
    for p, r in zip(preds, refs):
        tc += len(r); te += _edit(list(p), list(r))
    return te / max(tc, 1)

# ── IWSLT stamped.tsv parser ─────────────────────────────────
# Supports three layouts:
#   Layout A (shashwatup9k): id  wav  offset  duration  src_text  [tgt]   (5+ cols)
#   Layout B (minimal):      id  wav  src_text  [tgt]                      (col2 non-numeric)
#   Layout C (OdiaGenAI):    wav_filename  offset  duration                (3 cols, no text)

def parse_stamped_tsv(tsv_path):
    records = []
    with open(tsv_path, encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith('#'): continue
            cols = line.split('\t')
            if len(cols) < 3: continue

            # Layout C: exactly 3 cols, col[0] is a wav filename
            if len(cols) == 3 and (cols[0].endswith('.wav') or cols[0].endswith('.flac')):
                try:
                    records.append({
                        'id': Path(cols[0]).stem, 'wav': cols[0],
                        'offset': float(cols[1]), 'duration': float(cols[2]),
                        'src': '', 'tgt': '',
                    })
                    continue
                except ValueError:
                    pass

            try:    float(cols[2]); layout_a = True
            except: layout_a = False
            if layout_a and len(cols) >= 5:
                records.append({'id': cols[0], 'wav': cols[1],
                                 'offset': float(cols[2]), 'duration': float(cols[3]),
                                 'src': cols[4], 'tgt': cols[5] if len(cols)>5 else ''})
            else:
                records.append({'id': cols[0], 'wav': cols[1],
                                 'offset': 0., 'duration': -1.,
                                 'src': cols[2], 'tgt': cols[3] if len(cols)>3 else ''})
    return records

def _read_lines(path):
    if not os.path.exists(path): return []
    with open(path, encoding='utf-8') as f:
        return [l.strip() for l in f if l.strip()]

def load_txt_refs(txt_dir, split_key):
    """Load refs from a standard IWSLT txt/ directory."""
    src = (_read_lines(f'{txt_dir}/{split_key}.or')
           or _read_lines(f'{txt_dir}/{split_key}.odi')
           or _read_lines(f'{txt_dir}/txt.or')   # OdiaGenAI or/ directory
           or _read_lines(f'{txt_dir}/src'))
    hi  = _read_lines(f'{txt_dir}/{split_key}.hi') or _read_lines(f'{txt_dir}/dev.hi')
    en  = _read_lines(f'{txt_dir}/{split_key}.en') or _read_lines(f'{txt_dir}/dev.en')
    return src, hi, en

def load_split(data_root, split):
    """
    Load records for a split.  Handles both OdiaGenAI and shashwatup9k layouts.

    OdiaGenAI layout:
      test_set/  audio/  stamped.tsv(3-col)  or/txt.or  translations/dev.{hi,en}
      train/     audio/  txt.or
    """
    split_dir = os.path.join(data_root, split)

    # Prefer audio/ (OdiaGenAI), fall back to wav/ (standard IWSLT)
    wav_dir = os.path.join(split_dir, 'audio')
    if not os.path.isdir(wav_dir):
        wav_dir = os.path.join(split_dir, 'wav')

    tsv_path      = os.path.join(split_dir, 'stamped.tsv')
    or_txt_path   = os.path.join(split_dir, 'or', 'txt.or')    # test_set
    root_txt_path = os.path.join(split_dir, 'txt.or')            # train
    trans_dir     = os.path.join(split_dir, 'translations')
    txt_dir       = os.path.join(split_dir, 'txt')               # shashwatup9k

    is_odiagen = (os.path.isfile(or_txt_path) or os.path.isfile(root_txt_path)
                  or os.path.isdir(trans_dir))

    # Build records from TSV (or enumerate wav files if no TSV)
    if os.path.exists(tsv_path):
        records = parse_stamped_tsv(tsv_path)
    else:
        records = [
            {'id': Path(w).stem, 'wav': Path(w).name, 'offset': 0., 'duration': -1.,
             'src': '', 'tgt': ''}
            for w in sorted(glob.glob(f'{wav_dir}/*.wav'))
        ]

    # Attach text references
    if is_odiagen:
        src_l = _read_lines(or_txt_path) or _read_lines(root_txt_path)
        hi_l  = _read_lines(os.path.join(trans_dir, 'dev.hi'))
        en_l  = _read_lines(os.path.join(trans_dir, 'dev.en'))
        for i, r in enumerate(records):
            if i < len(src_l): r['src']    = src_l[i]
            if i < len(hi_l):  r['tgt_hi'] = hi_l[i]
            if i < len(en_l):  r['tgt_en'] = en_l[i]
    elif os.path.isdir(txt_dir):
        key = split.split('-')[0]
        src_l, hi_l, en_l = load_txt_refs(txt_dir, key)
        for i, r in enumerate(records):
            if i < len(src_l): r['src']    = src_l[i]
            if i < len(hi_l):  r['tgt_hi'] = hi_l[i]
            if i < len(en_l):  r['tgt_en'] = en_l[i]

    # Resolve absolute wav paths
    for r in records:
        wf = os.path.join(wav_dir, r['wav'])
        if not os.path.exists(wf) and not r['wav'].endswith('.wav'): wf += '.wav'
        r['wav_path'] = wf
    return records

# ── Inference ────────────────────────────────────────────────

def run_eval(data_root, split, checkpoint_path=None, sample_limit=None):
    print(f'\n--- Evaluating split: {split} ---')
    records = load_split(data_root, split)
    if sample_limit:
        records = records[:sample_limit]
    print(f'  Samples: {len(records)}')

    if checkpoint_path:
        state = torch.load(checkpoint_path, map_location=device)
        if isinstance(state, dict) and 'model_state' in state:
            state = state['model_state']
        model.load_state_dict(state)
        print(f'  Checkpoint loaded: {checkpoint_path}')
    model.eval()

    preds, refs_src, refs_hi = [], [], []
    BATCH = 8
    buf_wavs = []

    def flush_buf():
        wav_lens = torch.tensor([w.shape[0] for w in buf_wavs], dtype=torch.long)
        padded   = nn.utils.rnn.pad_sequence(buf_wavs, batch_first=True)
        padded, wl = padded.to(device), wav_lens.to(device)
        with torch.no_grad():
            lp, el = model(padded, wl)
        best = lp.argmax(-1).transpose(0,1)
        for j in range(best.shape[0]):
            preds.append(tokenizer.decode(best[j, :el[j]].tolist()))
        buf_wavs.clear()

    for i, rec in enumerate(records):
        refs_src.append(rec.get('src', ''))
        refs_hi.append(rec.get('tgt_hi', rec.get('tgt', '')))
        if not os.path.exists(rec['wav_path']):
            preds.append('')
            continue
        wav, sr = torchaudio.load(rec['wav_path'])
        wav = wav.mean(0)
        if rec['offset'] > 0 or rec['duration'] > 0:
            s = int(rec['offset'] * sr)
            e = s + int(rec['duration'] * sr) if rec['duration'] > 0 else None
            wav = wav[s:e]
        if sr != 16000:
            wav = torchaudio.transforms.Resample(sr, 16000)(wav)
        buf_wavs.append(wav)
        if len(buf_wavs) >= BATCH: flush_buf()
        if (i+1) % 200 == 0: print(f'  {i+1}/{len(records)}')
    if buf_wavs: flush_buf()

    wer_v = compute_wer(preds, refs_src) * 100 if any(refs_src) else None
    cer_v = compute_cer(preds, refs_src) * 100 if any(refs_src) else None

    print(f'\n{"="*55}')
    print(f'  Split : {split}')
    if wer_v is not None: print(f'  WER   : {wer_v:.2f}%')
    if cer_v is not None: print(f'  CER   : {cer_v:.2f}%')
    print(f'{"="*55}')
    print('\nSample predictions:')
    for i in range(min(5, len(preds))):
        print(f'  [REF] {refs_src[i][:70]}')
        print(f'  [HYP] {preds[i][:70]}\n')

    return {'split': split, 'wer': wer_v, 'cer': cer_v,
            'preds': preds, 'refs': refs_src, 'refs_hi': refs_hi}

print('Evaluation helpers loaded (OdiaGenAI layout).')


In [ ]:
BEST_CKPT  = os.path.join(CFG['output_dir'], 'best_model.pt')
IWSLT_ROOT = IWSLT_DIR   # set in clone cell above

all_results = {}

# ── Test set (main IWSLT 2026 benchmark) ─────────────────────
if os.path.isdir(os.path.join(IWSLT_ROOT, 'test_set')):
    all_results['test_set_finetuned'] = run_eval(
        IWSLT_ROOT, 'test_set', checkpoint_path=BEST_CKPT
    )

# ── Train set self-eval (sanity check) ───────────────────────
if os.path.isdir(os.path.join(IWSLT_ROOT, 'train')):
    # Use a small sample to avoid long runtime on Colab
    all_results['train_finetuned'] = run_eval(
        IWSLT_ROOT, 'train', checkpoint_path=BEST_CKPT, sample_limit=300
    )

# ── Baseline (no fine-tuning) on test_set ────────────────────
print('\n--- Baseline (no fine-tuning) ---')
from transformers import AutoModel as _AM
import torch.nn as nn

_base_enc = _AM.from_pretrained(
    'ai4bharat/indic-conformer-600m-multilingual', trust_remote_code=True
)
_base_model = CTCModel(_base_enc, len(tokenizer), 512).to(device)

# Temporarily swap the global model
_orig_model = model
globals()['model'] = _base_model

if os.path.isdir(os.path.join(IWSLT_ROOT, 'test_set')):
    all_results['test_set_baseline'] = run_eval(
        IWSLT_ROOT, 'test_set', checkpoint_path=None, sample_limit=200
    )

globals()['model'] = _orig_model  # restore fine-tuned model

# ── Summary table ─────────────────────────────────────────────
print('\n' + '='*55)
print('  SUMMARY  (OdiaGenAI/iwslt-odia-speech)')
print('='*55)
print(f'  {"Split":<25} {"WER":>8} {"CER":>8}')
print(f'  {"-"*43}')
for key, res in all_results.items():
    wer_str = f"{res['wer']:.2f}%" if res.get('wer') is not None else 'N/A'
    cer_str = f"{res['cer']:.2f}%" if res.get('cer') is not None else 'N/A'
    n       = len(res.get('preds', []))
    print(f'  {key:<25} {wer_str:>8} {cer_str:>8}  (n={n})')
print('='*55)

# Save results JSON
import json as _json
out_json = os.path.join(CFG['output_dir'], 'iwslt2026_eval_results.json')
with open(out_json, 'w', encoding='utf-8') as f:
    _json.dump({k: {'wer': v.get('wer'), 'cer': v.get('cer'),
                    'n': len(v.get('preds', []))}
                for k, v in all_results.items()}, f, indent=2, ensure_ascii=False)
print(f'\nSummary saved to: {out_json}')
